In [ ]:
import glob
import re
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter, gaussian_filter
from aicsimageio import AICSImage, types
from os import mkdir

from stardist.models import StarDist2D
from cellpose_omni import models

from matplotlib.colors import LinearSegmentedColormap
from skimage.measure import regionprops

## Overview
This is meant to loop through a directory of \*czi files and make pngs. Ideally, there are no-probe controls taken at the same intensity, with those czi's in a separate directory, from which the `read_controls` function can estimate intensities per channel and then be passed to `layoutImg` to do the actual running.

In [ ]:
def read_controls(imgs, mq=0.95, control_channels=[0,0,1], smooth=False):
    channel_max_uint16 = np.zeros((len(imgs),len(control_channels)))
    for enum,i in enumerate(imgs):
        img = AICSImage(i).data
        img = img[0,:,0,:,:]
        
        img = np.moveaxis(img, 0,2)
        print(img.shape)
        if img.shape[-1] > 3:
            continue
        if smooth:
            for chan in range(img.shape[-1]):
                img[:,:,chan] = median_filter(img[:,:,chan], size=3)
                #img[:,:,chan] = gaussian_filter(img[:,:,chan], sigma=5)
            #img = img / img.max(axis=(0,1))
        channel_max_uint16[enum,:] = np.quantile(img, mq, axis=(0,1))
        #img = np.where(img < 0, 0, img)
        
    channel_max_uint16 = channel_max_uint16 * control_channels
    #print(channel_max_uint16)
    return channel_max_uint16

imgs=glob.glob("hpg_H2_and_AcMoO4/well4_h2grass_24h_CONTROL/*.czi")#[1:2]
#print(imgs)
controls = read_controls(imgs, mq=0.99, control_channels=[1,0,0]).max(axis=0)

controls

#imgs=glob.glob("3768_2c_nodye_control/Image *.czi")#[1:2]
#controls3768 = read_controls(imgs).max(axis=0)


In [ ]:
#cmapCol=np.array(((0,0.5,1),(1,0,0)))  2 col
cmapCol=np.array(((1,0,0),(0,1,0),(1,1,1),(1,1,1)))  #sbyr hpg bf
#cmapCol=np.array(((1,1,1),(0,1,0),(1,0,1),(1,1,0)))  #bf
#cmapCol=np.array(((0,0.4,1),(0,1,0),(1,0,1),(1,1,0)))  #4 col

cmaps=[LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[0]], N=256),
       LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[1]], N=256),
       LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[2]], N=256),
       LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[3]], N=256)]

def layoutImg(i, mq=0.1, labs=['DAPI','Al488','cy3','Al647'], 
              cmaps=cmaps, cmapCol=cmapCol, scale=20, pngsize=16, save=True, folder='', suf='', 
              zstack=True, controls=[], quantmax=[], DIC=False, focuschans=[],
              omnipose=False, stardist=False, segstats=False):
    img = AICSImage(i)
    pix_um = img.physical_pixel_sizes.X
    #pix_um=0.07
    print(pix_um)
    #img = img.dask_data
    img = img.data
    #print(img.shape)
    if img.shape[1] == 1:
        print('one channel')
        return
    if img.shape[2] > 1:
        #print('zstack?')
        if zstack:
            #print(img.shape)
            img = np.max(img, axis=2)
            img = np.expand_dims(img, 2)
            #print(img.shape)
                
    #    return 'Z'
    #if img.shape[1] < 3:
    #    print('dapi?')
    #    return 'D'
    img = img[0,:,0,:,:]
    #print(img.shape)
    
    
    img = np.moveaxis(img, 0,2)
    if len(focuschans)>0:
        img=img[:,:,focuschans]
    if segstats:
        imgquant = img.copy()
    #print(img.shape)
    if len(controls) == 0:
        controls = np.repeat(0, img.shape[-1])
    #print(img.max(axis=(0,1)))
    minquant=np.quantile(img, mq, axis=(0,1))
    
    
    controls = np.where(controls == 0, minquant, controls)
    
    img = img - np.array(controls)
    maxquant = np.max(img, axis=(0,1))
    
    if DIC:
        pass
        #img = img - np.quantile(img, mq, axis=(0,1))[:-1]

    img = np.where(img < 0, 0, img)
    
    if len(quantmax) == 0:
        img = img / (1+ img.max(axis=(0,1)))
    else:
        whichmax=np.where(quantmax!=0)[0]
        #print([quantmax[x] if quantmax[x] > 0 else maxquant[x] for x in range(img.shape[-1])])
        img = img / np.where(quantmax == 0, maxquant, quantmax-controls[whichmax])
    #img = np.where(img < 0, 0, img)

    
    chans = img.shape[-1]
    
    #     if chans == 3:
    #         cmaps=[cmaps[x] for x in [0,2,3]]
    #         labs=[labs[x] for x in [0,2,3]]
    #         cmapCol=cmapCol[[0,2,3]]
    combchans=chans
    if DIC:
        combchans=combchans-1
    combimg = np.zeros((img.shape[0], img.shape[1],3))
    for chan in range(combchans):
        if DIC:
            chan=chan+1
        combimg = combimg + img[:,:,chan][:,:,np.newaxis] * cmapCol[np.newaxis,np.newaxis,chan]
    
    fig, axs=plt.subplots(nrows=2, ncols=2, figsize=(pngsize, pngsize))
        
    props = dict(linewidth=0, fc='black')
    
    axs[0,0].imshow(img[:,:,0], cmap=cmaps[0], vmin=0, vmax=1)
    axs[0,0].text(0.98,0.98, labs[0], fontsize=16, c=cmapCol[0], ha='right', va='top',transform = axs[0,0].transAxes, bbox=props)
    if chans > 1:
        axs[0,1].imshow(img[:,:,1], cmap=cmaps[1], vmin=0, vmax=1)
        axs[0,1].text(0.98,0.98, labs[1], fontsize=16, c=cmapCol[1], ha='right', va='top', transform = axs[0,1].transAxes, bbox=props)
        axs[1,0].axis('off')
        axs[1,1].axis('off')
    
    if chans == 2:
        axs[1,0].imshow(combimg)
        axs[1,0].text(0.98,0.98, labs[0], fontsize=16, c=cmapCol[0], ha='right', va='top',transform = axs[1,0].transAxes, bbox=props)
        axs[1,0].text(0.98,0.93, labs[1], fontsize=16, c=cmapCol[1], ha='right', va='top', transform = axs[1,0].transAxes, bbox=props)
    if chans == 3:
        axs[1,0].imshow(img[:,:,2], cmap=cmaps[2], vmin=0, vmax=1)
        axs[1,0].text(0.98,0.98, labs[2], fontsize=16, c=cmapCol[2], ha='right', va='top', transform = axs[1,0].transAxes, bbox=props)
        
        axs[1,1].imshow(combimg)
        axs[1,1].text(0.98,0.98, labs[0], fontsize=16, c=cmapCol[0], ha='right', va='top',transform = axs[1,1].transAxes, bbox=props)
        axs[1,1].text(0.98,0.93, labs[1], fontsize=16, c=cmapCol[1], ha='right', va='top', transform = axs[1,1].transAxes, bbox=props)
        axs[1,1].text(0.98,0.88, labs[2], fontsize=16, c=cmapCol[2], ha='right', va='top', transform = axs[1,1].transAxes, bbox=props)
    elif chans == 4 and not stardist:
        axs[1,0].imshow(img[:,:,2], cmap=cmaps[2], vmin=0, vmax=1)
        axs[1,0].text(0.98,0.98, labs[2], fontsize=16, c=cmapCol[2], ha='right', va='top', transform = axs[1,0].transAxes, bbox=props)
        axs[1,1].imshow(img[:,:,3], cmap=cmaps[3], vmin=0, vmax=1)
        axs[1,1].text(0.98,0.98, labs[3], fontsize=16, c=cmapCol[3], ha='right', va='top',transform = axs[1,1].transAxes, bbox=props)
    
    if chans > 2:
        axs[1,1].axvspan(xmin=.7*img.shape[0], xmax=.7*img.shape[0] + (scale/pix_um), ymin=0.07, ymax=0.09, fc='w')
        axs[1,1].text(.7*img.shape[0] + (scale/2/pix_um), 0.94*img.shape[0], str(scale) + ' µm', fontsize=16, c='w', ha='center', va='top')
    else:
        axs[1,0].axvspan(xmin=.7*img.shape[0], xmax=.7*img.shape[0] + (scale/pix_um), ymin=0.07, ymax=0.09, fc='w')
        axs[1,0].text(.7*img.shape[0] + (scale/2/pix_um), 0.94*img.shape[0], str(scale) + ' µm', fontsize=16, c='w', ha='center', va='top')

    axs[0,0].axis('off')
    axs[0,1].axis('off')

        
    if stardist:
        sdist = StarDist2D.from_pretrained('2D_versatile_fluo')
        masks, _ = sdist.predict_instances(img[:,:,1])
        if not segstats:
            axs[1,1].imshow(masks>1, cmap='gray')
            axs[1,1].text(0.98,0.98, 'Seg. mask', fontsize=16, c='w', ha='right', va='top',transform = axs[1,1].transAxes, bbox=props)
        else:
            regions = regionprops(masks)#measure.label(red_rois))
            intensity=np.zeros(len(regions))
            size=np.zeros(len(regions))
            #r=0
            for r,reg in enumerate(regions):
                pts = reg.coords
                intensity[r] = np.mean([imgquant[x,y,1] for x, y in pts])
                size[r] = reg.area
                #r+=1
            
            #print(intensity)
            vp = axs[1,1].violinplot(intensity,showmeans=False, showmedians=False, showextrema=False)
            [pc.set_facecolor(cmapCol[1]) for pc in vp['bodies']]
            axs[1,1].plot(np.random.normal(1, 0.04, size=len(intensity)), intensity, c=cmapCol[1], 
                          alpha=0.2, marker='.', linestyle='None')
            axs[1,1].axhline(y=controls[1],c='black', linestyle='--')
            
            axs[1,1].plot(np.random.normal(2, 0.04, size=len(intensity[intensity > controls[1]])), 
                          intensity[intensity>controls[1]], c=cmapCol[1], 
                          alpha=1, marker='.', linestyle='None')
            axs[1,1].plot(np.random.normal(2, 0.04, size=len(intensity[intensity < controls[1]])), 
                          intensity[intensity<controls[1]], c='blue', 
                          alpha=1, marker='.', linestyle='None')
            axs[1,1].text(0.5,0.98, 'N. HPG+ : ' + str(len(intensity[intensity > controls[1]])), 
                          fontsize=16, c=cmapCol[1], ha='center', va='top',transform = axs[1,1].transAxes, bbox=dict(linewidth=0, fc='white'))
            axs[1,1].text(0.5,0.93, 'N. HPG- : ' + str(len(intensity[intensity<controls[1]])), 
                          fontsize=16, c='blue', ha='center', va='top', transform = axs[1,1].transAxes, bbox=dict(linewidth=0, fc='white'))
            axs[1,1].text(0.5,0.88, str(round((pix_um*img.shape[0] * 0.001) * (pix_um*img.shape[1] * 0.001),6)) + 'mm^2', 
                          fontsize=16, c='black', ha='center', va='top', transform = axs[1,1].transAxes, bbox=dict(linewidth=0, fc='white'))
            axs[1,1].text(0.5,0.83, str(round((pix_um*img.shape[0]) * (pix_um*img.shape[1]),2)) + 'um^2', 
                          fontsize=16, c='black', ha='center', va='top', transform = axs[1,1].transAxes, bbox=dict(linewidth=0, fc='white'))
            axs[1,1].axis('on')


    elif omnipose:
        opose = models.CellposeModel(gpu=False, model_type='bact_fluor_cp')
        params = {'channels':[0,0], # always define this with the model
          'rescale': None, # upscale or downscale your images, None = no rescaling 
          'mask_threshold': 0, # erode or dilate masks with higher or lower values between -5 and 5 
          'flow_threshold': 0.9, # default is .4, but only needed if there are spurious masks to clean up; slows down output
          'transparency': True, # transparency in flow output
          'omni': True, # we can turn off Omnipose mask reconstruction, not advised 
          'cluster': True, # use DBSCAN clustering
          'resample': True, # whether or not to run dynamics on rescaled grid or original grid 
          'verbose': False, # turn on if you want to see more output 
          'tile': False, # average the outputs from flipped (augmented) images; slower, usually not needed 
          'niter': None, # default None lets Omnipose calculate # of Euler iterations (usually <20) but you can tune it for over/under segmentation 
          'augment': False, # Can optionally rotate the image and average network outputs, usually not needed 
          # 'affinity_seg': True, # new feature, stay tuned...
         }
        masks, flows, styles = opose.eval([img], **params)
        #print(masks)
        axs[1,1].imshow(flows[0][-1])#, cmap='gray')
        axs[1,1].text(0.98,0.98, 'Seg. mask', fontsize=16, c='w', ha='right', va='top',transform = axs[1,1].transAxes, bbox=props)
    
    plt.tight_layout()
    
    outf = re.sub('.[cz][[zv]i$', suf+'.png', i)
    outf= '-'.join(outf.split('/')[-4:])
    outf=folder + '/' + outf
    #print(outf)
    if save:
        try:
            plt.savefig(outf)    
        except FileNotFoundError:
            mkdir(folder)
            plt.savefig(outf)
    
    if chans == 4:
        plt.figure(figsize=(pngsize,pngsize))
        ax = plt.axes()
        plt.imshow(combimg)
        plt.axis('off')
        
        plt.text(0.98,0.98, labs[0], fontsize=16, c=cmapCol[0], ha='right', va='top',transform = ax.transAxes, bbox=props)
        plt.text(0.98,0.95, labs[1], fontsize=16, c=cmapCol[1], ha='right', va='top', transform = ax.transAxes, bbox=props)
        plt.text(0.98,0.92, labs[2], fontsize=16, c=cmapCol[2], ha='right', va='top', transform = ax.transAxes, bbox=props)
        plt.text(0.98,0.89, labs[3], fontsize=16, c=cmapCol[3], ha='right', va='top', transform = ax.transAxes, bbox=props)

        ax.axvspan(xmin=.7*img.shape[0], xmax=.7*img.shape[0] + (scale/pix_um), ymin=0.07, ymax=0.08, fc='w')
        ax.text(.7*img.shape[0] + (scale/2/pix_um), 0.94*img.shape[0], str(scale) + ' µm', fontsize=16, c='w', ha='center', va='top')
        
        if save:
            plt.savefig(re.sub('.png','_overlay.png',outf))
            
    if DIC:
        fig,axs = fig, axs=plt.subplots(nrows=1, ncols=1, figsize=(pngsize, pngsize))
        axs.imshow(img[:,:,0], cmap=cmaps[0], vmin=0, vmax=1)
        axs.text(0.98,0.98, labs[0], fontsize=16, c=cmapCol[0], ha='right', va='top',transform = axs.transAxes, bbox=props)
        axs.axvspan(xmin=.7*img.shape[0], xmax=.7*img.shape[0] + (scale/pix_um), ymin=0.07, ymax=0.09, fc='w')
        axs.text(.7*img.shape[0] + (scale/2/pix_um), 0.94*img.shape[0], str(scale) + ' µm', fontsize=16, c='w', ha='center', va='top')
        plt.axis('off')
        plt.tight_layout()
        
        if save:
            plt.savefig(re.sub('.png','_bf.png',outf))


## Example of how it is run

#### Arguments:
`i`= path to image file

`mq`= minimum quantile (percentile), default=0.1

`labs`= labels for the channels, should be exactly as long as the number of channels or longer. default \['DAPI','Al488','cy3','Al647'\] 

`cmaps`= colormaps as list of LinearSegmentedColormaps

`cmapCol`= tuple/list of rgb values, used for font labels

`scale`= sclaebar in microns, default=20

 ----
`pngsize`= size of png, default 16

`save`= Save images or just run? Default True (save)

`folder`= name of folder in which to save images, will create if does not exist. Default '' (no folder)

`suf`= suffix to attach to filename before the .png, default '' (no suffix)

---------
`controls`= 1D list the length of channels, e.g. [0,0,0,0] if 4 channel image. values are the maximum control value for thresholding. 0 means to disregard controls for that channel, e.g. [0,1083,1024,5000] does not apply control to DAPI (first channel) but gives channel-specific thresholds. Default is [] which is no controls.

`quantmax`= reverse of controls, same format. This normalizes the max value of all images to a constant number (usually the brightest pixel in any imgae). default [] which is no normalization.

-------
`omnipose`= run omnipose object detection, default False (don't run)

`stardist`= run stardist object detection, default False (don't run). This is usually better than omnipose

`segstats` = run statistics per ROI. Default False. Only relevant if omnipose or stardist is True.


In [ ]:
imgs=glob.glob("hpg_H2_and_AcMoO4/well10_*//*czi")
print(imgs)
#imgs=glob.glob("3768_2c_NR_WGA_4days_well3/Image 11.czi")#[1:2]

_ = [layoutImg(im, mq=0.0, labs=['HPG','Sybr','BF'], scale=10, DIC=False, #quantmax=[],
               save=False, 
               folder='anaeramoeba_hpg_2026Feb', suf='_HPGarb',
               #focuschans=[0,1],
             #controls=controls[[0,1]], #quantmax=maxes,
             pngsize=16, omnipose=False, stardist=False, segstats=False)#, controls=controls3767)
   for im in imgs]# if not re.search('tilescan', im)]

## Calculating per-channel max across a series of images

Quick way to generate a list of maximum values for use in `layoutImg`

#### Relevant arguments
`mq`= maximum quantile (1=highest pixel, 0.9=90th percentile). Default=1

`quant_channels` = mask of the channels to calculate max values for, must be the exact length of characters. For example [0,1,1] would calculate the maximum value of the second and third channels but not the first.

In [ ]:
def quantitative_max(imgs, mq=1, quant_channels=[0,0,1], pick_thresh=np.Inf, pick_channels=[]):
    channel_max_uint16 = np.zeros((len(imgs),len(quant_channels)))
    for enum,i in enumerate(imgs):
        img = AICSImage(i).data
        if img.shape[2] > 1:
            print('zstack?')
            img = np.max(img, axis=2)
            img = np.expand_dims(img, 2)
        img = img[0,:,0,:,:]
        
        img = np.moveaxis(img, 0,2)
        #print(img.shape)
        if img.shape[-1] > pick_thresh:
            img = img[:,:,pick_channels]
            print(img.shape)
        for chan in range(img.shape[-1]):
            img[:,:,chan] = median_filter(img[:,:,chan], size=3)
            #img[:,:,chan] = gaussian_filter(img[:,:,chan], sigma=5)
        #img = img / img.max(axis=(0,1))
        channel_max_uint16[enum,:] = np.quantile(img, mq, axis=(0,1))
        #img = np.where(img < 0, 0, img)
        
    channel_max_uint16 = channel_max_uint16 * quant_channels
    print(channel_max_uint16)
    return channel_max_uint16

imgs=glob.glob("hpg_H2_and_AcMoO4/well3_h2grass_6d/*.czi")[1:]
#print(imgs)
#maxes = quantitative_max(imgs).max(axis=0)
maxes = quantitative_max(imgs, mq=0.999, quant_channels=[1,1], 
                         pick_thresh=2, pick_channels=[0,1]).max(axis=0)

maxes


[Google doc with cell count math](https://docs.google.com/spreadsheets/d/1Zb9AwCgMGLsZO6dYqDlc8GN_K5-Pc65fu4Zn4knCjDI/edit?usp=sharing) 

requires cell counts (with `segstats` here) and mm^2 of the image (both computed here) and the diameter of filter area (hopefully known) and the various volumes and dilutions connecting original sample to amount actually loaded onto filter (hopefully in lab notebook)